<a href="https://colab.research.google.com/github/csrsustain/HVAC-Optimization-/blob/main/FCU_Overnight_Operation_(Heating).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# === INPUTS (from PEAK / TAB) ===
fcu_name = "........"       # -     Building and FCU number
airflow_ls = 200.0        # L/s   - airflow at overnight fan speed
sat = 28.0                # degC  - supply air temperature (warmer than RAT)
rat = 20.0                # degC  - return (or room) air temperature
fan_kw = 0.09             # kW    - FCU fan electrical power
hours = 12.0              # h     - overnight running hours per night
nights = 365              # -     - nights per year

boiler_efficiency = 0.85  # -     - seasonal boiler efficiency (0.80-0.95)
gas_per_kwh = 0.05        # GBP/kWh - gas tariff
elec_per_kwh = 0.25       # GBP/kWh - electricity tariff (for the fan)
co2_gas = 0.183           # kgCO2e/kWh - natural gas
co2_elec = 0.207          # kgCO2e/kWh - UK grid electricity

# === CONSTANTS ===
air_density = 1.204       # kg/m3
cp_air = 1.006            # kJ/kg.K

# === CALCULATION ===
delta_t = sat - rat
mass_flow = (airflow_ls / 1000.0) * air_density     # kg/s
kw_thermal = mass_flow * cp_air * delta_t           # kW heat delivered
kw_fuel = kw_thermal / boiler_efficiency            # kW gas input at the boiler

# Per night
kwh_gas_night = kw_fuel * hours
kwh_fan_night = fan_kw * hours
cost_night = kwh_gas_night * gas_per_kwh + kwh_fan_night * elec_per_kwh

# Per year
kwh_gas_year = kwh_gas_night * nights
kwh_fan_year = kwh_fan_night * nights
cost_gas_year = kwh_gas_year * gas_per_kwh
cost_fan_year = kwh_fan_year * elec_per_kwh
cost_year = cost_gas_year + cost_fan_year
co2_year = kwh_gas_year * co2_gas + kwh_fan_year * co2_elec

# === OUTPUT ===
W1, W2, W3 = 26, 14, 8      # label, value, unit column widths


def rule(left, mid, right, fill="\u2500"):
    print(left + fill * (W1 + 2) + mid + fill * (W2 + 2) + mid + fill * (W3 + 2) + right)


def row(label, value, unit=""):
    print(f"\u2502 {label:<{W1}} \u2502 {value:>{W2}} \u2502 {unit:<{W3}} \u2502")


def section(title):
    rule("\u251c", "\u253c", "\u2524")
    print(f"\u2502 {title:<{W1 + W2 + W3 + 6}} \u2502")
    rule("\u251c", "\u253c", "\u2524")


rule("\u250c", "\u252c", "\u2510")
print(f"\u2502 {'FCU OVERNIGHT HEATING COST - ' + fcu_name:<{W1 + W2 + W3 + 6}} \u2502")
print(f"\u2502 {'Sensible heat only (latent excluded)':<{W1 + W2 + W3 + 6}} \u2502")

section("INPUTS")
row("Airflow", f"{airflow_ls:,.1f}", "L/s")
row("Supply air temp (SAT)", f"{sat:,.1f}", "degC")
row("Return air temp (RAT)", f"{rat:,.1f}", "degC")
row("FCU fan power", f"{fan_kw:,.3f}", "kW")
row("Overnight hours", f"{hours:,.1f}", "h/night")
row("Nights per year", f"{nights:,}", "nights")
row("Boiler efficiency", f"{boiler_efficiency * 100:,.0f}", "%")
row("Gas tariff", f"{gas_per_kwh:,.3f}", "GBP/kWh")
row("Electricity tariff", f"{elec_per_kwh:,.3f}", "GBP/kWh")

if delta_t <= 0:
    section("RESULT")
    row("Delta T (SAT - RAT)", f"{delta_t:,.2f}", "degC")
    row("Status", "NOT HEATING", "-")
    rule("\u2514", "\u2534", "\u2518")
    print("\nSAT <= RAT: the unit is not heating, so there is no overnight heating cost.")
else:
    section("LOAD")
    row("Delta T (SAT - RAT)", f"{delta_t:,.2f}", "degC")
    row("Mass flow", f"{mass_flow:,.3f}", "kg/s")
    row("Heat delivered", f"{kw_thermal:,.2f}", "kW")
    row("Gas input at boiler", f"{kw_fuel:,.2f}", "kW")
    row("FCU fan draw", f"{fan_kw:,.2f}", "kW")

    section("ENERGY PER NIGHT")
    row("Gas", f"{kwh_gas_night:,.2f}", "kWh")
    row("Electricity (fan)", f"{kwh_fan_night:,.2f}", "kWh")
    row("Cost per night", f"{cost_night:,.2f}", "GBP")

    section("ANNUAL TOTALS")
    row("Gas energy", f"{kwh_gas_year:,.0f}", "kWh")
    row("Gas cost", f"{cost_gas_year:,.2f}", "GBP")
    row("Fan electricity", f"{kwh_fan_year:,.0f}", "kWh")
    row("Fan cost", f"{cost_fan_year:,.2f}", "GBP")
    row("TOTAL annual cost", f"{cost_year:,.2f}", "GBP")
    row("TOTAL annual CO2", f"{co2_year:,.0f}", "kgCO2e")
    rule("\u2514", "\u2534", "\u2518")
    print("\nThis is the avoidable cost if the FCU were switched off overnight.")

┌────────────────────────────┬────────────────┬──────────┐
│ FCU OVERNIGHT HEATING COST - ........                  │
│ Sensible heat only (latent excluded)                   │
├────────────────────────────┼────────────────┼──────────┤
│ INPUTS                                                 │
├────────────────────────────┼────────────────┼──────────┤
│ Airflow                    │          200.0 │ L/s      │
│ Supply air temp (SAT)      │           28.0 │ degC     │
│ Return air temp (RAT)      │           20.0 │ degC     │
│ FCU fan power              │          0.090 │ kW       │
│ Overnight hours            │           12.0 │ h/night  │
│ Nights per year            │            365 │ nights   │
│ Boiler efficiency          │             85 │ %        │
│ Gas tariff                 │          0.050 │ GBP/kWh  │
│ Electricity tariff         │          0.250 │ GBP/kWh  │
├────────────────────────────┼────────────────┼──────────┤
│ LOAD                                                  